In [26]:
import numpy as np
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity


ING_META_PATH = "ingredient_meta.csv"
ING_EMB_PATH  = "ingredient_embeddings.npy"

OUT_CSV  = "persona_vectors.csv"
OUT_NPY  = "persona_vectors.npy"
OUT_META = "persona_meta.csv"

CICA_WHITELIST_TOKENS = [
    "병풀", "센텔라", "centella", "asiatica",
    "마데카", "madeca", "madecass",
    "아시아티코", "asiatic",
    "아시아티코사이드", "asiaticoside",
    "마데카소사이드", "madecassoside",
    "아시아틱애씨드", "asiatic acid",
    "마데카식애씨드", "madecassic acid",
    "cica"
]


ingredient_meta = pd.read_csv(ING_META_PATH)
ingredient_embeddings = np.load(ING_EMB_PATH)

assert "ingredient_name" in ingredient_meta.columns
assert len(ingredient_meta) == ingredient_embeddings.shape[0]

EMBEDDING_DIM = ingredient_embeddings.shape[1]
print("성분 수:", len(ingredient_meta))
print("임베딩 차원:", EMBEDDING_DIM)

def norm_ing(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[\[\]\(\)\{\}]", " ", s)
    s = re.sub(r"[^0-9a-z가-힣\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

ingredient_meta["ingredient_name_norm"] = ingredient_meta["ingredient_name"].map(norm_ing)

ingredient_embedding_dict = {}
for idx, row in ingredient_meta.iterrows():
    key = row["ingredient_name_norm"]
    if key and key not in ingredient_embedding_dict:
        ingredient_embedding_dict[key] = ingredient_embeddings[idx]

print("정규화 성분 dict 크기:", len(ingredient_embedding_dict))

def _passes_cica_filter(matched_key_norm: str) -> bool:
    """
    '시카'는 substring 오염이 커서(예: 트라이에톡시카프릴릴실레인),
    병풀/센텔라 계열 토큰이 같이 있을 때만 True.
    """
    k = matched_key_norm
    return any(tok in k for tok in CICA_WHITELIST_TOKENS)

def find_group_embeddings(keyword: str, emb_dict: dict):
    """
    keyword: 상위 키워드(예: '병풀', '인삼', '나이아신아마이드', '시카')
    emb_dict: {ingredient_norm: embedding}
    return: (vecs, matched_keys)
    """
    kw = norm_ing(keyword)
    vecs, matched_keys = [], []

    for k, v in emb_dict.items():
        if kw in k:
            # '시카'는 오염 방지 필터 적용
            if kw == "시카" or kw == "cica":
                if not _passes_cica_filter(k):
                    continue
            vecs.append(v)
            matched_keys.append(k)

    return vecs, matched_keys

def build_group_vector(keywords, emb_dict, dim):
    """
    keywords: ["병풀","인삼"] 처럼 성분군 키워드 리스트
    return: (mean_vector, match_log)
    """
    all_vecs = []
    match_log = {}

    for kw in keywords:
        vecs, keys = find_group_embeddings(kw, emb_dict)
        match_log[kw] = keys
        all_vecs.extend(vecs)

    if all_vecs:
        return np.mean(np.stack(all_vecs), axis=0), match_log
    else:
        return np.zeros(dim, dtype=np.float32), match_log

persona_ingredient_preference = {
    "persona_1": ["히알루론산", "시카"],
    "persona_2": ["나이아신아마이드"],
    "persona_3": ["병풀", "인삼"]
}

persona_ingredient_vector = {}
persona_match_log = {}

for pid, keywords in persona_ingredient_preference.items():
    vec, matched = build_group_vector(keywords, ingredient_embedding_dict, EMBEDDING_DIM)
    persona_ingredient_vector[pid] = vec.astype(np.float32)
    persona_match_log[pid] = matched

for pid, vec in persona_ingredient_vector.items():
    print(pid, "ingredient vec norm:", float(np.linalg.norm(vec)))

print("\n[매칭 로그 요약(상위 5개만)]")
for pid, matched in persona_match_log.items():
    print(f"\n[{pid}]")
    for kw, keys in matched.items():
        print(f"  - {kw}: {len(keys)}개")
        for k in keys[:5]:
            print("    ", k)

persona_tone = {
    "persona_1": np.zeros(EMBEDDING_DIM, dtype=np.float32),
    "persona_2": np.zeros(EMBEDDING_DIM, dtype=np.float32),
    "persona_3": np.zeros(EMBEDDING_DIM, dtype=np.float32),
}


persona_risk_price_vector = {
    "persona_1": [1.0, 0.9, 0.8, 0.3],
    "persona_2": [0.3, 0.2, 0.2, 0.9],
    "persona_3": [0.4, 0.3, 0.1, 0.2],
}

persona_ids = sorted(
    set(persona_tone)
    & set(persona_ingredient_vector)
    & set(persona_risk_price_vector)
)

persona_final_vector = {}
for pid in persona_ids:
    final_vec = np.concatenate([
        persona_tone[pid],
        persona_ingredient_vector[pid],
        np.array(persona_risk_price_vector[pid], dtype=np.float32)
    ], axis=0)
    persona_final_vector[pid] = final_vec.astype(np.float32)

print("\n최종 벡터 차원:", len(next(iter(persona_final_vector.values()))))

rows = []
for pid in persona_ids:
    rows.append({
        "persona_id": pid,
        "tone_vector": persona_tone[pid].tolist(),
        "ingredient_vector": persona_ingredient_vector[pid].tolist(),
        "risk_price_vector": persona_risk_price_vector[pid],
        "final_vector": persona_final_vector[pid].tolist(),
    })

persona_df = pd.DataFrame(rows)
persona_df.to_csv(OUT_CSV, index=False)
print("CSV 저장:", OUT_CSV)

persona_ids_sorted = sorted(persona_final_vector.keys())
persona_matrix = np.stack([persona_final_vector[pid] for pid in persona_ids_sorted]).astype(np.float32)

np.save(OUT_NPY, persona_matrix)
pd.DataFrame({"persona_id": persona_ids_sorted}).to_csv(OUT_META, index=False)

print("NPY 저장:", OUT_NPY, persona_matrix.shape)
print("META 저장:", OUT_META)


vecs = np.stack([persona_final_vector[p] for p in persona_ids]).astype(np.float32)
print("\n페르소나 간 cosine similarity")
print(cosine_similarity(vecs))

v = persona_final_vector[persona_ids[0]]
print("\n[샘플 체크:", persona_ids[0], "]")
print("total dim:", v.shape)
print("tone norm:", float(np.linalg.norm(v[:EMBEDDING_DIM])))
print("ingredient norm:", float(np.linalg.norm(v[EMBEDDING_DIM:2*EMBEDDING_DIM])))
print("risk tail:", v[-4:])

성분 수: 3149
임베딩 차원: 768
정규화 성분 dict 크기: 3048
persona_1 ingredient vec norm: 22.77735137939453
persona_2 ingredient vec norm: 19.07308578491211
persona_3 ingredient vec norm: 20.689104080200195

[매칭 로그 요약(상위 5개만)]

[persona_1]
  - 히알루론산: 3개
     히알루론산
     히알루론산 난류 가금류
     히알루론산혼합제제 덱스트린
  - 시카: 0개

[persona_2]
  - 나이아신아마이드: 5개
     나이아신아마이드
     나이아신아마이드 0 11
     나이아신아마이드 20
     나이아신아마이드 40
     나이아신아마이드 50

[persona_3]
  - 병풀: 12개
     2-헥산다이올 펜틸렌글라이콜 카프릴릴글라이콜 캐모마일꽃추출물 병풀추출물 마데카소사이드
     병풀꽃 잎 줄기추출물
     병풀꽃 잎 줄기추출물 플루이드 정제수
     병풀꽃 잎 줄기추출물 25 ppm
     병풀꽃 잎 줄기추출물 3 125 ppm
  - 인삼: 13개
     인삼가루
     인삼꽃추출물
     인삼뿌리세포추출물
     인삼뿌리프로토플라스트
     인삼수

최종 벡터 차원: 1540
CSV 저장: persona_vectors.csv
NPY 저장: persona_vectors.npy (3, 1540)
META 저장: persona_meta.csv

페르소나 간 cosine similarity
[[1.         0.878198   0.97531086]
 [0.878198   0.9999999  0.90628296]
 [0.97531086 0.90628296 0.99999976]]

[샘플 체크: persona_1 ]
total dim: (1540,)
tone norm: 0.0
ingredient norm: 22.77735137939453
risk ta